I originally took the direction of a more traditional mmi2x2, with straight waveguides and taper, having a fixed gap, and was sweeping both mmi length and width.

In order to avoid overlapping monitors I had to restrict their width, and there were definitely many parameters to tune.

I thus switched to having s-bend waveguides leading to the mmi region, fixed gap (the minimum specified by the task), and I only varied the length of the mmi.

In order to allocate more resources to the simulation I used mpi with meep, running the script as:
`mpirun -np {number of processors} python mmi2x2.py`

Once I did this for a range of lengths in 3D simulations I extracted the *S_parameters_mmi2x2_with_bends_3D.csv*

Then, settling to one length that balances between loss and splitting ratio I did another run of the mmi2x2.py script with spectrum extraction, saving wavelengths and scattering matrix info in *S_parameters_mmi2x2_with_bends_3p15.csv*.

## Post-processing: 3D length sweep

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("S_parameters_mmi2x2_with_bends_3D.csv").drop_duplicates()

# Extract unique axis values
lengths = np.sort(df["mmi_length (um)"].unique())

# Prepare empty grids
S31_grid = np.zeros(len(lengths))
S41_grid = np.zeros(len(lengths))

# Fill the grids
for _, row in df.iterrows():
    L = row["mmi_length (um)"]
    j = np.where(lengths == L)[0][0]

    S31_grid[j] = row["S31"]
    S41_grid[j] = row["S41"]

# Convert to dB
S31_dB = 20 * np.log10(np.abs(S31_grid))
S41_dB = 20 * np.log10(np.abs(S41_grid))


In [ ]:
plt.plot(lengths, S31_dB, marker='o')
plt.plot(lengths, S41_dB, marker='o')
plt.xlabel("MMI Length (µm)")
plt.ylabel("S-parameter (dB)")
plt.title("S-parameters vs MMI Length for MMI 2x2 with Bends")
plt.legend(["S31", "S41"])
plt.grid()
plt.show()

In [ ]:
import numpy as np

# condition where both S31_dB and S41_dB > -4
goal = -4.2
condition = (S31_dB > goal) & (S41_dB > goal)

# get indices where condition is True
indices = np.argwhere(condition)

# print the corresponding geometry values
for idx in indices.flatten():

    print(f"mmi_length = {lengths[idx]:.2f} µm, "
          f"S31_dB = {S31_dB[idx]:.2f}, S41_dB = {S41_dB[idx]:.2f}, loss = {-10*np.log10(S31_grid[idx]**2 + S41_grid[idx]**2):.2f} dB")


I am choosing L = 3.15um as it is a compromise between loss and splitting

## Post-processing for chosen MMI-region length

In [ ]:
import re

df = pd.read_csv("S_parameters_mmi2x2_with_bends_3D_3p15.csv")

# Extract unique axis values
wavelengths = df["wavelength (um)"]

def parse_array(s):
    # remove outer brackets
    s = s.strip()[1:-1]
    # replace whitespace between numbers with commas
    s = re.sub(r"\s+", ",", s)
    return np.fromstring(s, sep=",")

# Convert all columns
wl  = parse_array(df["wavelength (um)"][0])
S11 = parse_array(df["S11"][0])
S21 = parse_array(df["S21"][0])
S31 = parse_array(df["S31"][0])
S41 = parse_array(df["S41"][0])

transmitance = S31**2 + S41**2
port3_ratio = S31**2 / transmitance
port4_ratio = S41**2 / transmitance

# Convert to dB
S11_dB = 20 * np.log10(np.abs(S11))
S21_dB = 20 * np.log10(np.abs(S21))
S31_dB = 20 * np.log10(np.abs(S31))
S41_dB = 20 * np.log10(np.abs(S41))

Loss_dB = -10 * np.log10(S31**2 + S41**2)

### Scattering Parameters Spectrum

In [ ]:
plt.plot(wl, S31_dB, label="S31", color='blue')
plt.plot(wl, S41_dB, label="S41", color='orange')
plt.xlabel("Wavelength (µm)")
plt.ylabel("S-parameter (dB)")
plt.title("S-parameters vs Wavelength for MMI 2x2 with Bends (L=3.15 µm)")
plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.plot(wl, S11_dB, label="S31", color='green')
plt.plot(wl, S21_dB, label="S41", color='red')
plt.plot(wl, S31_dB, label="S31", color='blue')
plt.plot(wl, S41_dB, label="S41", color='orange')
plt.xlabel("Wavelength (µm)")
plt.ylabel("S-parameter (dB)")
plt.title("S-parameters vs Wavelength for MMI 2x2 with Bends (L=3.15 µm)")
plt.legend()
plt.grid()
plt.show()

### Loss Spectrum

In [ ]:
plt.plot(wl, Loss_dB, label="Loss")
plt.xlabel("Wavelength (µm)")
plt.ylabel("Loss (dB)")
plt.title("Insertion Loss vs Wavelength for MMI 2x2 with Bends (L=3.15 µm)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.plot(wl, port3_ratio, label="Port 3 Ratio")
plt.plot(wl, port4_ratio, label="Port 4 Ratio")
plt.xlabel("Wavelength (µm)")
plt.ylabel("Power Ratio")
plt.title("Splitting Ratios vs Wavelength for MMI 2x2 with Bends (L=3.15 µm)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
port3_ratio[int(len(wl)/2)], port4_ratio[int(len(wl)/2)], Loss_dB[int(len(wl)/2)]

**At 1550nm we have:**
- **A 53.7/46.3 splitting ratio**
- **Loss = 0.43 dB**

## Visualize Device's cross-section

In [ ]:
from mmi2x2 import mmi2x2_bend_waveguides
import meep as mp
import numpy as np
import matplotlib.pyplot as plt

wavelength = 1.55
fcen = 1/wavelength
df = 0.2*fcen
n_Si = 3.48  # silicon
n_SiO2 = 1.444  # SiO2

In [ ]:
THREE_D = False
sim, cell, mode1, mode2, mode3, mode4, src_volume, resolution, pml_layers, geometry = mmi2x2_bend_waveguides(mmi_length=3.15)

In [ ]:
eps_data = sim.get_array(center=mp.Vector3(), size=cell.size, component=mp.Dielectric)
plt.figure()
plt.imshow(eps_data.transpose(), interpolation='spline36', cmap='binary')
plt.axis('off')
plt.show()